# 04 · Inferencia para Kaggle

Lee las series DICOM de test, promedia uno o más checkpoints y crea `submission.csv`. Está pensado para copiarse al notebook de Kaggle junto con `src/` y los pesos. No usa internet; el tiempo depende del número de modelos y estudios, con límite total de 9 horas.

In [ ]:
# CONFIG
RUN_NAME = "kaggle_inference_v1"
DATA_ROOT = "/workspace/data/raw"
TEST_SERIES_DIR = "/workspace/data/raw/test_series"
BACKBONE_PATH = "/workspace/models/dinov2-small"
CHECKPOINTS = ["/workspace/users/ana/runs/dinov2s_baseline_v1/fold_0/best.pt"]
OUTPUT_PATH = "submission.csv"


In [ ]:
from pathlib import Path
import sys
import pandas as pd

repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo))
from src.knee.infer import predict_submission

data = Path(DATA_ROOT)
test = pd.read_csv(data / "test.csv")
test_series = pd.read_csv(data / "test_series.csv")
sample = pd.read_csv(data / "sample_submission.csv")
print(f"Estudios de test: {len(test):,}; checkpoints: {len(CHECKPOINTS)}")
for checkpoint in CHECKPOINTS:
    assert Path(checkpoint).exists(), f"Falta el checkpoint: {checkpoint}"


In [ ]:
submission = predict_submission(
    studies=test.StudyInstanceUID,
    series_csv=test_series,
    raw_series_root=TEST_SERIES_DIR,
    checkpoint_paths=CHECKPOINTS,
    sample_submission=sample,
    backbone_path=BACKBONE_PATH,
)
submission.to_csv(OUTPUT_PATH, index=False)
display(submission.head())
print(f"Guardado: {Path(OUTPUT_PATH).resolve()}")


In [ ]:
assert list(submission.columns) == list(sample.columns)
assert len(submission) == len(sample)
assert submission.iloc[:, 1:].notna().all().all()
assert submission.iloc[:, 1:].apply(lambda column: column.between(0, 1).all()).all()
print("Submission validada: filas, columnas y probabilidades correctas.")
